# Preprocessing CLAHE & Gaussian Denoising

## 1. Instalasi & Import Library

In [ ]:
!pip install opencv-python-headless matplotlib numpy scikit-image tqdm -q


In [ ]:
import shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from skimage import img_as_float, img_as_ubyte
from skimage.filters import gaussian


---
## 2. Konfigurasi Path Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# INPUT_BASE = hasil Stage 1, OUTPUT_BASE = hasil Stage 2 (CLAHE + Gaussian)
INPUT_BASE  = Path("/content/drive/MyDrive/Tugas Akhir/FINAL_dataset_roi_preprocessed_test")
OUTPUT_BASE = Path("/content/drive/MyDrive/Tugas Akhir/FINAL_GC_test")

SPLITS  = ["train", "val", "test"]
CLASSES = ["no_tumor", "tumor"]

# Hyperparameter CLAHE
CLAHE_CLIP_LIMIT = 2.0     # Batas amplifikasi kontras
CLAHE_TILE_GRID  = (8, 8)  # Ukuran grid tile lokal (piksel)

# Hyperparameter Gaussian Denoising
GAUSSIAN_SIGMA = 0.8       # Deviasi standar kernel Gaussian

# Buat struktur folder output (split x class + masks)
for split in SPLITS:
    for cls in CLASSES:
        (OUTPUT_BASE / split / cls).mkdir(parents=True, exist_ok=True)
        (OUTPUT_BASE / split / cls / "masks").mkdir(parents=True, exist_ok=True)


---
## 3. Implementasi CLAHE

In [ ]:
def apply_clahe(image_bgr: np.ndarray,
                clip_limit: float = CLAHE_CLIP_LIMIT,
                tile_grid: tuple  = CLAHE_TILE_GRID) -> np.ndarray:
    # CLAHE hanya diterapkan ke channel L (LAB) agar warna asli (a, b) tak terdistorsi
    lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe_obj = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    l_equalized = clahe_obj.apply(l_channel)

    lab_equalized = cv2.merge([l_equalized, a_channel, b_channel])
    result_bgr = cv2.cvtColor(lab_equalized, cv2.COLOR_LAB2BGR)
    return result_bgr


---
## 4. Implementasi Gaussian Denoising

In [ ]:
def apply_gaussian_denoise(image_bgr: np.ndarray,
                           sigma: float = GAUSSIAN_SIGMA) -> np.ndarray:
    img_float      = img_as_float(image_bgr)
    denoised_float = gaussian(img_float, sigma=sigma, channel_axis=-1)
    denoised_uint8 = img_as_ubyte(np.clip(denoised_float, 0.0, 1.0))  # clip cegah overflow sebelum ke uint8
    return denoised_uint8


---
## 5. Pipeline Preprocessing (CLAHE → Gaussian)

In [ ]:
def preprocess_stage2(image_path: Path,
                      clahe_clip: float = CLAHE_CLIP_LIMIT,
                      clahe_tile: tuple  = CLAHE_TILE_GRID,
                      gauss_sigma: float = GAUSSIAN_SIGMA) -> np.ndarray:
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(f"Citra tidak ditemukan: {image_path}")

    img_clahe    = apply_clahe(img, clip_limit=clahe_clip, tile_grid=clahe_tile)
    img_denoised = apply_gaussian_denoise(img_clahe, sigma=gauss_sigma)
    return img_denoised


---
## 6. Visualisasi Hasil pada Sampel Citra

In [ ]:
def visualize_pipeline(image_path: Path) -> None:
    img_original = cv2.imread(str(image_path))
    img_clahe    = apply_clahe(img_original)
    img_final    = apply_gaussian_denoise(img_clahe)

    titles = ["Input Asli", "Setelah CLAHE", "Setelah Gaussian Denoise"]
    images = [img_original, img_clahe, img_final]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, title, img in zip(axes, titles, images):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.axis('off')

    fig.suptitle(f"CLAHE (clip={CLAHE_CLIP_LIMIT}, tile={CLAHE_TILE_GRID})  →  "
                f"Gaussian Denoise (σ={GAUSSIAN_SIGMA})", fontsize=12)
    plt.tight_layout()
    plt.savefig("preprocessing_comparison.png", dpi=150, bbox_inches='tight')
    plt.show()


# Jalankan visualisasi pada sampel pertama yang ditemukan
sample_found = False
for split in SPLITS:
    for cls in CLASSES:
        cls_dir = INPUT_BASE / split / cls
        if cls_dir.exists():
            samples = list(cls_dir.glob("*.png"))
            if samples:
                print(f"Sampel uji: {samples[0]}")
                visualize_pipeline(samples[0])
                sample_found = True
                break
    if sample_found:
        break

if not sample_found:
    print("Peringatan: tidak ada citra ditemukan di INPUT_BASE.")


---
## 7. Pemrosesan Batch Seluruh Dataset

In [ ]:
def preprocess_mask(mask_path: Path, output_path: Path) -> None:
    # Mask = ground truth, disalin apa adanya (tanpa CLAHE/Gaussian)
    if not mask_path.exists():
        raise FileNotFoundError(f"Mask tidak ditemukan: {mask_path}")
    shutil.copy2(str(mask_path), str(output_path))


def batch_preprocess_dataset(input_base: Path,
                              output_base: Path,
                              splits: list,
                              classes: list) -> dict:
    stats = {
        split: {
            cls: {'success': 0, 'failed': 0, 'mask_success': 0, 'mask_failed': 0}
            for cls in classes
        }
        for split in splits
    }

    for split in splits:
        for cls in classes:
            cls_input    = input_base  / split / cls
            cls_output   = output_base / split / cls
            masks_input  = cls_input   / "masks"
            masks_output = cls_output  / "masks"

            if not cls_input.exists():
                print(f"  Peringatan: {split}/{cls} tidak ditemukan, dilewati.")
                continue

            image_paths = list(cls_input.glob("*.png"))

            for img_path in tqdm(image_paths, desc=f"    {split}/{cls}", unit="img"):
                try:
                    processed = preprocess_stage2(img_path)
                    out_path  = cls_output / img_path.name
                    cv2.imwrite(str(out_path), processed)
                    stats[split][cls]['success'] += 1
                except Exception as e:
                    print(f"\n     Gagal memproses {img_path.name}: {e}")
                    stats[split][cls]['failed'] += 1

            if masks_input.exists():
                mask_paths = list(masks_input.glob("*.png"))
                if mask_paths:
                    masks_output.mkdir(parents=True, exist_ok=True)
                    for mask_path in tqdm(mask_paths, desc=f"    {split}/{cls}/masks", unit="mask"):
                        try:
                            out_mask_path = masks_output / mask_path.name
                            preprocess_mask(mask_path, out_mask_path)
                            stats[split][cls]['mask_success'] += 1
                        except Exception as e:
                            print(f"\n     Gagal menyalin mask {mask_path.name}: {e}")
                            stats[split][cls]['mask_failed'] += 1

    return stats


results = batch_preprocess_dataset(INPUT_BASE, OUTPUT_BASE, SPLITS, CLASSES)

print("\n" + "=" * 55)
print(" RINGKASAN HASIL")
print("=" * 55)
total_ok = total_fail = 0
total_mask_ok = total_mask_fail = 0
for split in SPLITS:
    for cls in CLASSES:
        stat = results[split][cls]
        ok, fail     = stat['success'], stat['failed']
        m_ok, m_fail = stat['mask_success'], stat['mask_failed']
        total_ok        += ok
        total_fail      += fail
        total_mask_ok   += m_ok
        total_mask_fail += m_fail
        status = "OK" if (fail == 0 and m_fail == 0) else "WARN"
        print(f"  {status:<4} {split:<6}/{cls:<12} →  "
              f"citra: {ok:>5} OK / {fail:>3} gagal  |  "
              f"mask: {m_ok:>5} OK / {m_fail:>3} gagal")

print("-" * 55)
print(f"     Total citra →  {total_ok:>5} berhasil  |  {total_fail:>3} gagal")
print(f"     Total mask  →  {total_mask_ok:>5} disalin  |  {total_mask_fail:>3} gagal")
print(f"\nOutput tersimpan di: {OUTPUT_BASE.resolve()}")


---
## 8. Verifikasi Hasil Akhir

In [ ]:
def show_final_samples(output_base: Path,
                        splits: list,
                        classes: list,
                        n_per_class: int = 3) -> None:
    # Pakai split pertama yang sudah punya data output
    display_split = None
    for split in splits:
        if any(
            (output_base / split / cls).exists() and
            list((output_base / split / cls).glob("*.png"))
            for cls in classes
        ):
            display_split = split
            break

    if display_split is None:
        print("Peringatan: tidak ada data output ditemukan. Jalankan batch processing terlebih dahulu.")
        return

    fig, axes = plt.subplots(len(classes), n_per_class,
                             figsize=(n_per_class * 4, len(classes) * 4))

    for row, cls in enumerate(classes):
        cls_dir = output_base / display_split / cls
        samples = list(cls_dir.glob("*.png"))[:n_per_class]

        for col in range(n_per_class):
            ax = axes[row][col] if len(classes) > 1 else axes[col]
            if col < len(samples):
                img = cv2.imread(str(samples[col]))
                ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                ax.axis('off')
                if col == 0:
                    ax.set_ylabel(cls, fontsize=12, fontweight='bold',
                                  rotation=90, labelpad=10, va='center')
            else:
                ax.axis('off')

    fig.suptitle(
        f"Sampel Hasil Preprocessing Akhir — Split: {display_split}\n"
        "(CLAHE + Gaussian Denoising)",
        fontsize=14, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    plt.savefig("final_samples_grid.png", dpi=150, bbox_inches='tight')
    plt.show()


show_final_samples(OUTPUT_BASE, SPLITS, CLASSES)
